# Notebook 9 — Réentraînement complet, labels v2 (nomenclature AHA/ACCF/HRS)

Les règles d'étiquetage suivent maintenant les recommandations AHA/ACCF/HRS (code ≥ 2026.11.20).
Tous les modèles de l'article sont réentraînés avec ces labels. Les noms commencent par `L2_`
pour ne jamais reprendre par erreur un ancien entraînement.

**Réglages :** *Accelerator* **GPU T4** ; *Internet* **On** ;
*Input* : (1) la **nouvelle version** du dataset du code, (2) la sortie du notebook 1 (cache),
(3) la sortie du notebook 8 (contient `lsas_codes.csv`).

**Pour continuer après 12 h :** *Add Input* → *Your Work* → sortie de **ce notebook** (dernière version),
puis *Save & Run All*. Répétez jusqu'à **« FILE D'ATTENTE TERMINÉE »**.
L'ordre de la liste est l'ordre de priorité : les 3 premiers suffisent pour les résultats principaux.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
import json
meta = glob.glob("/kaggle/input/**/cache/mimic/meta.csv", recursive=True)
assert meta, "Cache introuvable : ajoutez la sortie du notebook 1 comme Input."
CACHE = os.path.dirname(os.path.dirname(meta[0]))
_i = json.load(open(f"{CACHE}/mimic/info.json"))
if not _i.get("complete", True):
    print(f"ATTENTION : cache MIMIC incomplet ({_i['n_done']}/{_i['n']}). Terminez le notebook 1 avant d'entraîner.")
# Copie des caches sur le disque local de la machine. Lire des ECG au hasard directement dans
# /kaggle/input est extrêmement lent (le 1er test : ~25 min pour 12 000 ECG) ; une copie
# séquentielle prend quelques minutes, ensuite chaque époque est rapide.
import time
LOCAL = "/tmp/cache_local"          # hors /kaggle/working : non sauvegardé, recréé à chaque session
def copy_cache(name):
    src, dst = f"{CACHE}/{name}", f"{LOCAL}/{name}"
    os.makedirs(dst, exist_ok=True)
    t0, total = time.time(), 0
    for f in sorted(os.listdir(src)):
        s_, d_ = os.path.join(src, f), os.path.join(dst, f)
        if os.path.exists(d_) and os.path.getsize(d_) == os.path.getsize(s_):
            continue
        shutil.copyfile(s_, d_); total += os.path.getsize(d_)
    dt = time.time() - t0
    print(f"  {name:6s} {total / 1e9:5.1f} Go copiés en {dt / 60:4.1f} min ({total / 1e6 / max(dt, 1e-6):.0f} Mo/s)")
need = sum(os.path.getsize(f"{CACHE}/{d}/signals.npy") for d in ["mimic", "ptbxl", "lsas"])
free = shutil.disk_usage("/tmp").free
print(f"Disque local libre : {free / 1e9:.0f} Go, besoin : {need / 1e9:.1f} Go")
if free > 1.15 * need:
    for d in ["mimic", "ptbxl", "lsas"]:
        copy_cache(d)
    SRC = LOCAL
else:
    print("ATTENTION : pas assez de place pour la copie locale ; lecture directe (lente) depuis l'Input")
    SRC = CACHE
# Labels MIMIC recalculés avec les règles actuelles du code (quelques secondes)
MIMIC = "/tmp/cache_labels/mimic"
!python scripts/audit_labels.py --cache {SRC}/mimic --relabel --link --quiet --out {MIMIC}
# LSAS : labels v2 (liste SNOMED-CT complète) recalculés depuis lsas_codes.csv (sortie du notebook 8)
_codes = sorted(glob.glob("/kaggle/input/**/lsas_codes.csv", recursive=True)
                + glob.glob("/kaggle/working/lsas_codes.csv"))
if not _codes:
    # pas de sortie du notebook 8 : lecture directe des en-têtes LSAS sur PhysioNet (10-15 min, Internet On).
    # Le fichier est gardé dans la sortie de ce notebook : les reprises suivantes le retrouvent.
    print("lsas_codes.csv absent des Inputs : lecture des en-têtes LSAS sur PhysioNet (10-15 min)...")
    !python -c "import sys; sys.path.insert(0, 'scripts'); from lsas_code_check import fetch_codes; fetch_codes().to_csv('/kaggle/working/lsas_codes.csv', index=False)"
    _codes = glob.glob("/kaggle/working/lsas_codes.csv")
if not _codes:
    raise SystemExit("ARRÊT : codes LSAS impossibles à obtenir. Vérifiez que Internet est sur On.")
print("Codes LSAS :", _codes[0])
LSAS = "/tmp/cache_labels/lsas"
!python scripts/relabel_cache.py --dataset lsas --cache {SRC}/lsas --out {LSAS} --codes {_codes[0]}
PATHS = (f"data.train_cache={MIMIC} "
         f"data.eval_sets.mimic_internal.cache={MIMIC} "
         f"data.eval_sets.mimic_val.cache={MIMIC} data.eval_sets.mimic_val.split=val "
         f"data.eval_sets.ptbxl_external.cache={SRC}/ptbxl "
         f"data.eval_sets.lsas_external.cache={LSAS}")

In [ ]:
import ecgfusion; assert ecgfusion.__version__ >= "2026.11.20", "Ancienne version du code en Input"
V1, V2 = "configs/kaggle.yaml", "configs/kaggle_v2.yaml"
AMP = "data.signal_norm=fixed"
EXPERIMENTS = [
    # --- priorité 1 : modèle final, 3 graines (résultats principaux, gate, calibration, dérivations)
    ("L2_final_s42",   V2, [AMP]),
    ("L2_final_s1",    V2, [AMP, "seed=1"]),
    ("L2_final_s2",    V2, [AMP, "seed=2"]),
    # --- priorité 2 : ablations clés
    ("L2_sans_abandon", V2, [AMP, "train.p_drop_text=0", "train.p_drop_features=0", "train.p_drop_leads=0"]),
    ("L2_bce_s42",     V2, [AMP, "train.loss=bce"]),
    ("L2_bce_s1",      V2, [AMP, "train.loss=bce", "seed=1"]),
    ("L2_bce_s2",      V2, [AMP, "train.loss=bce", "seed=2"]),
    ("L2_signal_seul", V2, [AMP, "train.train_scenario=signal_only", "train.val_scenarios=[signal_only]",
                            "model.text_encoder=light",
                            "eval.scenarios=[signal_only,limb6_signal_only,leadI_signal_only]"]),
    # --- priorité 3 : normalisation par dérivation (z-score) et masquage partiel du texte (v1)
    ("L2_z_s42",       V2, []),
    ("L2_z_s1",        V2, ["seed=1"]),
    ("L2_z_s2",        V2, ["seed=2"]),
    ("L2_z_bce",       V2, ["train.loss=bce"]),
    ("L2_v1_masquage", V1, []),
]
SESSION_HOURS = 11.0
EVAL_HOURS = 0.75

In [ ]:
import json, shlex, time
T0 = time.time()
WORK = "/kaggle/working/runs"
os.makedirs(WORK, exist_ok=True)

def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600

def status(d):
    p = f"{d}/status.json"
    return json.load(open(p)) if d and os.path.exists(p) else None

def previous(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/runs/{name}/config.yaml", recursive=True))
    return os.path.dirname(hits[0]) if hits else None

def copy_run(src, dst):
    shutil.copytree(src, dst, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("last.pt") if (status(src) or {}).get("finished") else None)

paths = shlex.split(PATHS)
report, stopped = [], False
for name, conf, extra in EXPERIMENTS:
    run, prev = f"{WORK}/{name}", previous(name)
    if prev and not os.path.exists(run):
        copy_run(prev, run)
        print(f"[{name}] repris depuis {prev}")
    st = status(run)
    done_eval = os.path.exists(f"{run}/eval/summary.csv")
    # run évalué avant le réétiquetage LSAS (labels v2) : on remplace les labels LSAS à partir
    # des prédictions sauvegardées (CPU, quelques minutes), puis l'analyse est refaite
    if (done_eval and globals().get("LSAS", "").startswith("/tmp/cache_labels")
            and not os.path.exists(f"{run}/eval/labels_v2_lsas_external.done")):
        !python scripts/relabel_eval.py --run {run} --set lsas_external --cache {LSAS}
    done_ana = os.path.exists(f"{run}/analysis/gating.csv")
    if st and st["finished"] and done_eval and done_ana:
        report.append((name, "terminée")); continue

    # 1) entraînement (nouveau ou reprise)
    if not (st and st["finished"]):
        budget = hours_left() - EVAL_HOURS
        if budget < 0.5:
            stopped = True; report.append((name, "pas commencée (temps)")); break
        resume = ["--resume", run] if os.path.exists(f"{run}/last.pt") else []
        cmd = shlex.join(["python", "scripts/train.py", "--config", conf, "--max_hours", f"{budget:.3f}",
                          *resume, "--set", *paths, f"output_dir={run}", *extra])
        print(f"\n===== {name} : entraînement ({'reprise' if resume else 'nouveau'}), budget {budget:.1f} h")
        !{cmd}
        st = status(run)
        if not (st and st["finished"]):
            stopped = True; report.append((name, f"entraînement en cours (époque {(st or {}).get('last_epoch', '?')})")); break

    # 2) évaluation
    if not os.path.exists(f"{run}/eval/summary.csv"):
        if hours_left() < EVAL_HOURS:
            stopped = True; report.append((name, "entraînée, évaluation à faire")); break
        cmd = shlex.join(["python", "scripts/evaluate.py", "--run", run, "--set", *paths, *extra])
        print(f"\n===== {name} : évaluation")
        !{cmd}
        if globals().get("LSAS", "").startswith("/tmp/cache_labels") and os.path.exists(f"{run}/eval/summary.csv"):
            open(f"{run}/eval/labels_v2_lsas_external.done", "w").write("ok\n")

    # 3) analyse (CPU)
    if not os.path.exists(f"{run}/analysis/gating.csv"):
        !python scripts/analyze_preds.py --run {run}
    if os.path.exists(f"{run}/last.pt"):
        os.remove(f"{run}/last.pt")   # inutile une fois terminé (place disque)
    report.append((name, "terminée"))

print("\n" + "=" * 60)
for name, s_ in report:
    print(f"  {name:15s} {s_}")
left = [n for n, _, _ in EXPERIMENTS if n not in {r for r, s_ in report if s_ == 'terminée'}]
print("=" * 60)
print("FILE D'ATTENTE TERMINÉE" if not left else
      f"À REPRENDRE ({len(left)} expérience(s) restante(s)) : ajoutez la sortie de cette version en Input et relancez.")
print("=" * 60)

## Récapitulatif des expériences terminées (AUROC macro)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220)
rows = []
ref = sorted(glob.glob("/kaggle/input/**/runs/fusion_v2/eval/summary.csv", recursive=True))
sources = ([("fusion_v2 (graine 42)", ref[0])] if ref else []) + \
          [(n, f"{WORK}/{n}/eval/summary.csv") for n, _, _ in EXPERIMENTS]
for name, f in sources:
    if os.path.exists(f):
        s_ = pd.read_csv(f); s_["run"] = name; rows.append(s_)
if rows:
    tab = pd.concat(rows)
    keep = ["full", "no_text", "signal_only"]
    display(tab[tab.scenario.isin(keep)].pivot_table(index="run", columns=["eval_set", "scenario"],
                                                     values="macro_auroc").round(3))
    seeds = tab[tab.run.str.startswith(("fusion_v2", "v2_seed"))]
    if seeds.run.nunique() >= 2:
        print("Graines (moyenne ± écart-type) :")
        g = seeds[seeds.scenario.isin(keep)].groupby(["eval_set", "scenario"])["macro_auroc"]
        display((g.mean().round(4).astype(str) + " ± " + g.std().round(4).astype(str)).unstack())
else:
    print("Aucune expérience terminée pour l'instant.")